# Hindi Spelling Accuracy Improvement

This notebook implements a pipeline to identify correctly vs. incorrectly spelled words in a large Hindi conversational dataset.

The core problem is to distinguish between legitimate words (including English words transcribed in Devanagari) and obvious transcriptionErrors. Once errors are identified, only the corresponding audio segments need re-transcription, saving significant resources.

## 1. Setup and Dependencies

We use standard data processing and text manipulation libraries.

In [1]:
import os
import pandas as pd
import numpy as np
import re
from collections import Counter
from tqdm.auto import tqdm

## 2. Data Loading and Preprocessing

The dataset contains approximately 177,000 unique words. We start by cleaning raw transcription artifacts like punctuation and symbols.

In [2]:
csv_path = 'dataset/Unique Words Data - Sheet1.csv'
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
else:
    df = pd.DataFrame({'word': ['है', 'तो', 'मम', 'कंपयूट्टर']})

def clean_word(word):
    if not isinstance(word, str): return ''
    word = re.sub(r'[\.\,|।"\(\)\[\]\{\}\!\?\*]', '', word)
    return word.strip()

df['cleaned_word'] = df['word'].apply(clean_word)
df = df[df['cleaned_word'] != ''].copy()

## 3. Spell Checking Logic

Our approach combines three layers of validation:
1. **Vocabulary Match**: Checking against a known list of high-frequency correct words.
2. **Grammatical Constraints**: Identifying invalid Devanagari character sequences (e.g., duplicate matras).
3. **ASR Artifact Detection**: Identifying standard fillers/disfluencies.

In [3]:
core_vocab = {
    'है', 'तो', 'में', 'जी', 'हैं', 'भी', 'के', 'नहीं', 'कि', 'वो', 'और', 'से', 'जो', 'हो', 'मतलब',
    'हां', 'हम', 'की', 'एक', 'ही', 'का', 'आप', 'को', 'ये', 'था', 'बहुत', 'मैं', 'कुछ', 'अच्छा',
    'बिल्कुल', 'बात', 'पर', 'थे', 'अगर', 'पे', 'ऐसा', 'या', 'मुझे', 'लिए', 'रहा', 'रहे', 'आ', 'अपने',
    'स्कूल', 'दोस्त', 'फेमस', 'फ्रेंड्स', 'मार्केट', 'कंप्यूटर', 'जॉब', 'मोबाइल', 'इंटरव्यू', 'प्रॉब्लम'
}

def classify_word(word):
    if word in core_vocab:
        return 'correct spelling', 'high', 'Matched common vocabulary'

    if re.match(r'^(ह्म्म|हम्म|अह|उह|अरे|ओह|हल्लो|यस|नो)$', word):
        return 'correct spelling', 'high', 'Standard conversational filler'

    if re.search(r'[\u093e-\u094c][\u093e-\u094c]', word):
        return 'incorrect spelling', 'high', 'Invalid vowel mark sequence'

    if re.search(r'[\u0905-\u0914][\u093e-\u094d]', word):
        return 'incorrect spelling', 'high', 'Invalid vowel mark on standalone vowel'

    if re.match(r'^[\u093e-\u094d]', word):
        return 'incorrect spelling', 'high', 'Word starts with vowel mark'

    if re.match(r'^[\u0900-\u097f]+$', word):
        return 'correct spelling', 'low', 'Grammatically plausible Devanagari'

    return 'incorrect spelling', 'high', 'Non-Devanagari characters'

tqdm.pandas()
results = df['cleaned_word'].progress_apply(classify_word)
df['label'] = results.apply(lambda x: x[0])
df['confidence'] = results.apply(lambda x: x[1])
df['reason'] = results.apply(lambda x: x[2])

  0%|          | 0/177478 [00:00<?, ?it/s]

## 4. Evaluation and Review

We analyze the 'low confidence' words to find where the system might be unreliable (e.g., proper nouns).

In [4]:
low_conf_sample = df[df['confidence'] == 'low'].sample(n=min(10, len(df)), random_state=1)
low_conf_sample[['cleaned_word', 'label', 'reason']]

,cleaned_word,label,reason
116628,रास्तानी,correct spelling,Grammatically plausible Devanagari
9682,लागा,correct spelling,Grammatically plausible Devanagari
115987,फीरे,correct spelling,Grammatically plausible Devanagari
18027,आजमाया,correct spelling,Grammatically plausible Devanagari
7804,केंद्रित,correct spelling,Grammatically plausible Devanagari
73161,किसीकिसी,correct spelling,Grammatically plausible Devanagari
132658,प्रेंडिंग,correct spelling,Grammatically plausible Devanagari
113044,दूरव्यवहार,correct spelling,Grammatically plausible Devanagari
49095,ज्वैलर्स,correct spelling,Grammatically plausible Devanagari
49310,एक्सली,correct spelling,Grammatically plausible Devanagari


## 5. Final Output Generation

We calculate the total number of correct words and export the final classification.

In [5]:
final_count = len(df[df['label'] == 'correct spelling'])
print(f"Total Unique Correct Spelled Words: {final_count}")

df[['word', 'label']].to_csv('Categorized_Unique_Words.csv', index=False)

Total Unique Correct Spelled Words: 170531
